In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# === 🌊 升级版扭曲函数：复合波浪 (Compound Wave) ===
def warp_text_map_advanced(img_arr, amplitude=6.0, period=30.0):
    """
    现在的扭曲不再是单一的正弦波，而是叠加了高频噪音。
    这会让线条看起来更像“手抖”画出来的，而不是电脑生成的。
    """
    h, w = img_arr.shape
    map_x = np.zeros((h, w), np.float32)
    map_y = np.zeros((h, w), np.float32)
    
    # 随机相位，保证每次生成的扭曲都不一样
    phase_x = np.random.rand() * 10
    phase_y = np.random.rand() * 10
    
    for y in range(h):
        for x in range(w):
            # X轴扭曲：一个大波浪 + 一个高频小抖动
            wave_main = amplitude * np.sin(2 * np.pi * y / period + phase_x)
            wave_noise = (amplitude * 0.5) * np.sin(2 * np.pi * y / (period * 0.3)) # 高频抖动
            
            # Y轴扭曲：同样叠加
            wave_y_main = amplitude * np.cos(2 * np.pi * x / period + phase_y)
            
            map_x[y, x] = x + wave_main + wave_noise
            map_y[y, x] = y + wave_y_main
            
    return cv2.remap(img_arr, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

# === ✍️ 绘制函数 ===
def draw_spaced_text(canvas, text, font, scale, thickness, spacing):
    h, w = canvas.shape
    total_text_width = 0
    char_widths = []
    
    for char in text:
        (cw, ch), _ = cv2.getTextSize(char, font, scale, thickness)
        char_widths.append(cw)
        total_text_width += cw
    
    total_width_with_spacing = total_text_width + (len(text) - 1) * spacing
    current_x = (w - total_width_with_spacing) // 2
    (_, th), _ = cv2.getTextSize("A", font, scale, thickness)
    base_y = (h + th) // 2
    
    for i, char in enumerate(text):
        cv2.putText(canvas, char, (current_x, base_y), font, scale, 255, thickness)
        current_x += char_widths[i] + spacing

def generate_thick_twisted_illusion(text_v, text_h):
    # ================= 🔧 参数大改区 =================
    W, H = 1600, 1600
    TEXT_STRIP_H = 150    # 加高文字条，容纳更粗的字
    
    # 间距调整：因为字变粗了，需要稍微加大间距防止粘连
    LETTER_SPACING = 45   
    
    # 扭曲参数：力度加大
    WARP_AMP = 6.0        # 扭曲幅度更大
    WARP_FREQ = 45.0      # 波长
    
    # 笔触参数
    NUM_STROKES = 35000   # 笔触更密集，填满粗字
    JITTER_AMT = 6        # 抖动更剧烈，边缘更毛糙
    # =================================================

    print(f"正在生成【加粗+狂乱版】错觉图...")

    font = cv2.FONT_HERSHEY_SIMPLEX
    
    # --- 1. 制作 Map A (竖向) ---
    map_v_small = np.zeros((TEXT_STRIP_H, W), dtype=np.uint8)
    
    scale, thickness = 1, 2
    while True:
        tot_w = sum([cv2.getTextSize(c, font, scale, thickness)[0][0] for c in text_v]) + (len(text_v)-1)*LETTER_SPACING
        (_, th), _ = cv2.getTextSize("A", font, scale, thickness)
        
        if tot_w > W * 0.92 or th > TEXT_STRIP_H * 0.9:
            scale = max(0.5, scale - 0.1)
            # 【关键修改】这里改成 4.0，让字体显著变粗
            thickness = max(1, int(scale * 4.0)) 
            break
        scale += 0.1
        thickness = int(scale * 4.0) # 保持加粗比例
    
    draw_spaced_text(map_v_small, text_v, font, scale, thickness, LETTER_SPACING)
    
    # 使用高级复合扭曲
    map_v_small = warp_text_map_advanced(map_v_small, amplitude=WARP_AMP, period=WARP_FREQ)
    map_v = cv2.resize(map_v_small, (W, H), interpolation=cv2.INTER_LINEAR)


    # --- 2. 制作 Map B (横向) ---
    map_h_small = np.zeros((TEXT_STRIP_H, W), dtype=np.uint8)
    
    scale, thickness = 1, 2
    while True:
        tot_w = sum([cv2.getTextSize(c, font, scale, thickness)[0][0] for c in text_h]) + (len(text_h)-1)*LETTER_SPACING
        (_, th), _ = cv2.getTextSize("A", font, scale, thickness)
        
        if tot_w > W * 0.92 or th > TEXT_STRIP_H * 0.9:
            scale = max(0.5, scale - 0.1)
            thickness = max(1, int(scale * 4.0)) # 加粗
            break
        scale += 0.1
        thickness = int(scale * 4.0) # 加粗
        
    draw_spaced_text(map_h_small, text_h, font, scale, thickness, LETTER_SPACING)
    
    # 使用高级复合扭曲
    map_h_small = warp_text_map_advanced(map_h_small, amplitude=WARP_AMP, period=WARP_FREQ)
    
    map_h_rotated = cv2.rotate(map_h_small, cv2.ROTATE_90_CLOCKWISE)
    map_h = cv2.resize(map_h_rotated, (W, H), interpolation=cv2.INTER_LINEAR)


    # --- 3. 绘制素描 ---
    final_img = Image.new('RGB', (W, H), color='white')
    draw = ImageDraw.Draw(final_img)
    
    # 绘制竖向笔触 (V)
    for _ in range(NUM_STROKES):
        rx = np.random.randint(0, W)
        ry = np.random.randint(0, H)
        
        pixel_val = map_v[ry, rx]
        # 阈值调低，只要有一点灰度就画，保证粗字填得满
        if pixel_val > 40: 
            length = np.random.randint(40, 140) 
            # 笔触宽度也稍微加粗一点点，范围 [3, 7]
            width = np.random.randint(3, 7) 
            col = np.random.randint(0, 60) # 颜色更深
            
            jitter_x = np.random.randint(-JITTER_AMT, JITTER_AMT)
            draw.line([(rx + jitter_x, ry), (rx + jitter_x, ry+length)], fill=(col,col,col), width=width)
            
    # 绘制横向笔触 (H)
    for _ in range(NUM_STROKES):
        rx = np.random.randint(0, W)
        ry = np.random.randint(0, H)
        
        pixel_val = map_h[ry, rx]
        if pixel_val > 40: 
            length = np.random.randint(40, 140)
            width = np.random.randint(3, 7)
            col = np.random.randint(0, 60)
            
            jitter_y = np.random.randint(-JITTER_AMT, JITTER_AMT)
            draw.line([(rx, ry + jitter_y), (rx+length, ry + jitter_y)], fill=(col,col,col), width=width)

    # 背景杂线 (更加细碎，像铅笔稿的底色)
    for _ in range(18000):
        rx, ry = np.random.randint(0, W), np.random.randint(0, H)
        if map_v[ry, rx] < 30 and map_h[ry, rx] < 30:
            if np.random.rand() > 0.5:
                l = np.random.randint(5, 20)
                d = np.random.randint(-4, 4)
                # 颜色更浅一点，突出前景的粗字
                draw.line([(rx, ry), (rx+d, ry+l)], fill=(210,210,210), width=1)
            else:
                l = np.random.randint(5, 20)
                d = np.random.randint(-4, 4)
                draw.line([(rx, ry), (rx+l, ry+d)], fill=(210,210,210), width=1)

    # --- 4. 模拟视图 ---
    final_arr = np.array(final_img)
    
    # 充电口视角
    preview_h = int(H * 0.04) 
    sim_v = cv2.resize(final_arr, (W, preview_h), interpolation=cv2.INTER_AREA)
    sim_v = cv2.resize(sim_v, (W, H), interpolation=cv2.INTER_NEAREST)
    
    # 侧边视角
    rotated_img = cv2.rotate(final_arr, cv2.ROTATE_90_COUNTERCLOCKWISE)
    sim_h = cv2.resize(rotated_img, (W, preview_h), interpolation=cv2.INTER_AREA)
    sim_h = cv2.resize(sim_h, (W, H), interpolation=cv2.INTER_NEAREST)

    return final_img, sim_v, sim_h

# ================= 运行区 =================
TEXT_V = "IHATEPKUICS" 
TEXT_H = "ILOVEPKUCV"

illusion, sim_v, sim_h = generate_thick_twisted_illusion(TEXT_V, TEXT_H)

# 展示
plt.figure(figsize=(18, 6))
plt.subplot(1, 3, 1)
plt.imshow(illusion)
plt.title("Normal View (Bold & Messy)", fontsize=12)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(sim_v)
plt.title(f"Bottom View", fontsize=12)
plt.xlabel(TEXT_V)
plt.xticks([])
plt.yticks([])

plt.subplot(1, 3, 3)
plt.imshow(sim_h)
plt.title(f"Side View", fontsize=12)
plt.xlabel(TEXT_H)
plt.xticks([])
plt.yticks([])

plt.show()

# 保存
save_name = f"illusion_bold_{TEXT_V}_{TEXT_H}.png"
illusion.save(save_name)
print(f"✅ 已保存: {save_name}")
print(f"👉 升级完成：字体已大幅加粗，且增加了不规则的复合扭曲。")